In [88]:
# ================================================================
# STAGE 9 — ADVANCED FEATURE ENGINEERING
# ================================================================

import numpy as np
import pandas as pd

print("=" * 70)
print("STAGE 9 — ADVANCED FEATURE ENGINEERING")
print("=" * 70)

def engineer_features(df):

    X = df.copy()

    # ------------------------------------------------
    # Amount transformations
    # ------------------------------------------------

    X["LogAmount"] = np.log1p(X["Amount"])

    # ------------------------------------------------
    # Time transformations
    # ------------------------------------------------

    X["TimeHours"] = X["Time"] / 3600

    X["HourSin"] = np.sin(
        2 * np.pi * X["Hour"] / 24
    )

    X["HourCos"] = np.cos(
        2 * np.pi * X["Hour"] / 24
    )

    # ------------------------------------------------
    # PCA component statistics
    # ------------------------------------------------

    v_cols = [
        c for c in X.columns
        if c.startswith("V")
        and c[1:].isdigit()
    ]

    X["V_Mean"] = X[v_cols].mean(axis=1)
    X["V_Std"] = X[v_cols].std(axis=1)
    X["V_Min"] = X[v_cols].min(axis=1)
    X["V_Max"] = X[v_cols].max(axis=1)

    X["V_AbsMean"] = X[v_cols].abs().mean(axis=1)
    X["V_AbsMax"] = X[v_cols].abs().max(axis=1)
    X["V_L2"] = np.sqrt(
        (X[v_cols] ** 2).sum(axis=1)
    )

    # ------------------------------------------------
    # Amount interactions
    # ------------------------------------------------

    X["AmountPerHour"] = (
        X["Amount"] /
        (X["TimeHours"] + 1)
    )

    X["Amount_V_AbsMean"] = (
        X["Amount"] * X["V_AbsMean"]
    )

    return X


# ------------------------------------------------
# Apply to ORIGINAL split data
# ------------------------------------------------

X_train_fe = engineer_features(X_train)
X_val_fe = engineer_features(X_val)

print("Original features :", X_train.shape[1])
print("New features      :", X_train_fe.shape[1])
print("Added features    :", X_train_fe.shape[1] - X_train.shape[1])

print("\nNew feature names:")
print([
    c for c in X_train_fe.columns
    if c not in X_train.columns
])

# ------------------------------------------------
# Train XGBoost
# ------------------------------------------------

xgb_fe = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="aucpr",
    random_state=42,
    n_jobs=-1
)

xgb_fe.fit(
    X_train_fe,
    y_train
)

val_proba_fe = xgb_fe.predict_proba(X_val_fe)[:, 1]
val_pred_fe = (val_proba_fe >= 0.5).astype(int)

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

tn, fp, fn, tp = confusion_matrix(
    y_val,
    val_pred_fe
).ravel()

print("\n" + "=" * 70)
print("FEATURE ENGINEERING RESULTS")
print("=" * 70)

print(confusion_matrix(y_val, val_pred_fe))

print(
    classification_report(
        y_val,
        val_pred_fe,
        digits=4
    )
)

print(
    "ROC-AUC :",
    roc_auc_score(y_val, val_proba_fe)
)

print(
    "PR-AUC  :",
    average_precision_score(y_val, val_proba_fe)
)

STAGE 9 — ADVANCED FEATURE ENGINEERING
Original features : 31
New features      : 44
Added features    : 13

New feature names:
['LogAmount', 'TimeHours', 'HourSin', 'HourCos', 'V_Mean', 'V_Std', 'V_Min', 'V_Max', 'V_AbsMean', 'V_AbsMax', 'V_L2', 'AmountPerHour', 'Amount_V_AbsMean']

FEATURE ENGINEERING RESULTS
[[28324     1]
 [   13    35]]
              precision    recall  f1-score   support

           0     0.9995    1.0000    0.9998     28325
           1     0.9722    0.7292    0.8333        48

    accuracy                         0.9995     28373
   macro avg     0.9859    0.8646    0.9165     28373
weighted avg     0.9995    0.9995    0.9995     28373

ROC-AUC : 0.9730560458958517
PR-AUC  : 0.8145817175279666
